In [1]:
from plantdata import load_classes, read_manifest
from pathlib import Path
from training import run_experiment

SPLITS_DIR = Path("splits_v2")
CACHE_DIR = Path("cache_256")
CLASS_IDS, CROPS, crop_mask = load_classes(SPLITS_DIR)
manifest = read_manifest(CACHE_DIR)

BASE = dict(depth=18, freeze="none", augment=True, optimiser="sgd", lr=1e-3, weight_decay=1e-4,
            batch_size=64, epochs=15, patience=5, seed=42)
CHECKS = [
    ("R19", dict(class_weights="balanced"), "class-weighted loss (rare classes count more; compare R05)"),
    ("R20", dict(depth=50, lr=3e-4),        "fairness check: ResNet-50 at lower LR 3e-4 (compare R18)"),
]

for run_id, change, note in CHECKS:
    run_experiment(run_id, note, manifest, CACHE_DIR, CLASS_IDS, CROPS, crop_mask, **{**BASE, **change})

R19: class-weighted loss (rare classes count more; compare R05) | 11,186,772 trainable weights
  epoch  1 | lr 1.00e-03 | train loss 0.531 acc 0.829 | val_lab F1-16 0.900 | val_wild F1-16 0.447 * | 101 s
  epoch  2 | lr 9.89e-04 | train loss 0.156 acc 0.955 | val_lab F1-16 0.944 | val_wild F1-16 0.463 * | 33 s
  epoch  3 | lr 9.57e-04 | train loss 0.103 acc 0.968 | val_lab F1-16 0.958 | val_wild F1-16 0.465 * | 36 s
  epoch  4 | lr 9.05e-04 | train loss 0.085 acc 0.973 | val_lab F1-16 0.956 | val_wild F1-16 0.475 * | 34 s
  epoch  5 | lr 8.35e-04 | train loss 0.069 acc 0.980 | val_lab F1-16 0.964 | val_wild F1-16 0.462 | 34 s
  epoch  6 | lr 7.50e-04 | train loss 0.062 acc 0.981 | val_lab F1-16 0.962 | val_wild F1-16 0.467 | 34 s
  epoch  7 | lr 6.55e-04 | train loss 0.054 acc 0.984 | val_lab F1-16 0.973 | val_wild F1-16 0.460 | 34 s
  epoch  8 | lr 5.52e-04 | train loss 0.052 acc 0.983 | val_lab F1-16 0.970 | val_wild F1-16 0.469 | 34 s
  epoch  9 | lr 4.48e-04 | train loss 0.047 acc 

In [2]:
import pandas as pd

s = pd.read_csv("results/runs_summary.csv").set_index("run_id")
s.loc[["R05", "R15", "R16", "R19"],
      ["batch_size", "class_weights", "best_epoch", "val_wild_f1_16", "val_wild_f1_20",
       "val_wild_late_blight_recall", "val_wild_accuracy", "train_minutes"]].round(3)


,batch_size,class_weights,best_epoch,val_wild_f1_16,val_wild_f1_20,val_wild_late_blight_recall,val_wild_accuracy,train_minutes
run_id,,,,,,,,
R05,64,none,4,0.469,0.488,0.720,0.511,6.081
R15,32,none,2,0.474,0.490,0.802,0.509,5.168
R16,16,none,5,0.473,0.484,0.675,0.498,7.090
R19,64,balanced,4,0.475,0.494,0.699,0.512,6.274
